## Segmentation and quantification
This script uses Cellpose to segment droplets from OIR files, generate NumPy masks, and quantify fluorescence intensities.  
The analysis environment can be reproduced using the following command. Replace envname with the desired environment name and specify the correct path to the .yml file:  
mamba env create -n envname -f D:\temp\cellpose3.yml

### Import libraries and define functions

In [5]:
# Import required libraries
import os

# Limit OpenMP to one thread to avoid a known KMeans memory leak on Windows systems using MKL. 
# This must be set before importing NumPy or scikit-learn.
os.environ["OMP_NUM_THREADS"] = "1"

import warnings
import urllib.parse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import imagej
from cellpose import models, utils
from scipy import ndimage as ndi
from skimage import filters

# Test data
main_dir = os.path.join(os.path.curdir, "Test data")

# Full dataset
#main_dir = os.path.join(os.path.curdir, "260426 Data") 

# Initialize PyImageJ and load the Cellpose model.
ij = imagej.init('sc.fiji:fiji')
model = models.CellposeModel(model_type='cyto3')

def open_oir(image_path):
    """Open an OIR image using PyImageJ and convert it to a NumPy array."""
    encoded_path = urllib.parse.quote(image_path)
    image_java = ij.io().open(encoded_path)
    image_oir = np.array(ij.py.from_java(image_java))
    return image_oir

def normalize_image(image):
    """Normalize image pixel intensities to the range 0-1 for segmentation."""
    image = image.astype(np.float32)
    normalized_channel= (image - image.min()) / (image.max() - image.min())
    return normalized_channel

def merge_rgb(image_oir, red_channel, green_channel, blue_channel):
    """Generate an RGB composite from selected image channels."""
    def get_channel(image_oir, idx):
        if idx is None:
            return np.zeros(image_oir.shape[:2], dtype=np.float32)
        return normalize_image(image_oir[:, :, idx])

    ch_red = get_channel(image_oir, red_channel)
    ch_green = get_channel(image_oir, green_channel)
    ch_blue = get_channel(image_oir, blue_channel)
    composite_image = np.stack([ch_red, ch_green, ch_blue], axis=2)
    return composite_image

def filter_masks_by_size(masks, size=100):
    """Remove segmented objects smaller than the specified number of pixels."""
    count = 0
    masks_filtered = masks.copy()
    for label in np.unique(masks):
        if np.sum(masks[:, :] == label) < size:
            count += 1
            masks_filtered[masks_filtered == label] = 0
        else:
            masks_filtered[masks_filtered == label] = label - count
    return masks_filtered

def divide_green_or_not(data, masks, masks_inner=None):
    """
    Classify droplets into high- and low-FAM-fluorescence groups using k-means clustering of log-transformed FAM fluorescence intensities,
    and generate corresponding masks.
    """
    log_green_values = np.log(data['FAM_Fl_Int'].values)
    kmeans = KMeans(n_clusters=2, n_init=10, random_state=0).fit(log_green_values.reshape(-1, 1))
    centers = kmeans.cluster_centers_.flatten()  # Fluorescence-intensity centers of the two clusters.

    # The cluster with the higher center is designated as the Green group.
    bigger_idx = np.argmax(centers) 
    smaller_idx = 1 - bigger_idx
    data_green = data.loc[kmeans.labels_ == bigger_idx].copy()
    labels_green = data_green['label'].values
    data_nogreen = data.loc[kmeans.labels_ == smaller_idx].copy()
    labels_nogreen = data_nogreen['label'].values

    # Relabel objects consecutively from 1 within each group.
    masks_green = np.zeros_like(masks)
    masks_nogreen = np.zeros_like(masks)
    masks_green_inner = None
    masks_nogreen_inner = None
    if masks_inner is not None:
        masks_green_inner = np.zeros_like(masks_inner)
        masks_nogreen_inner = np.zeros_like(masks_inner)
    
    for i, label in enumerate(labels_green):
        data_green.loc[data_green['label'] == label, 'label'] = i + 1
        masks_green[masks == label] = i + 1
        if masks_inner is not None:
            masks_green_inner[masks_inner == label] = i+1
    for i, label in enumerate(labels_nogreen):
        data_nogreen.loc[data_nogreen['label'] == label, 'label'] = i + 1
        masks_nogreen[masks == label] = i + 1
        if masks_inner is not None:
            masks_nogreen_inner[masks_inner == label] = i+1
            
    return data_green, data_nogreen, masks_green, masks_nogreen, masks_green_inner, masks_nogreen_inner

def fill_holes_including_edge(image):
    """
    Fill enclosed regions, including objects that contact the image edges. 
    The image is processed separately along the horizontal and vertical boundaries because enclosing all four edges 
    may simultaneously cause the entire image background to be treated as a single hole. 
    Objects contacting image corners may not be completely filled.
    """
    filled = normalize_image(image)
    filled = np.pad(filled, pad_width=1, mode='constant', constant_values=0)
    
    # Temporarily close the left and right boundaries before hole filling.
    filled[:, 0] = 1
    filled[:, -1] = 1
    filled = ndi.binary_fill_holes(filled[:, :]> filters.threshold_otsu(filled[:, :]))
    filled[:, 0] = 0
    filled[:, -1] = 0
    
    # Temporarily close the top and bottom boundaries before hole filling.
    filled[0, :] = 1
    filled[-1, :] = 1
    filled = ndi.binary_fill_holes(filled)
    filled[0, :] = 0
    filled[-1, :] = 0

    # Remove the added padding.
    filled = filled[1:-1, 1:-1]
    return filled

def divide_red_or_blue(image_oir, data, masks, masks_inner=None):
    """Classify droplets into "Red" and "Blue" groups based on ATTO 565- or ATTO 647-labeled liposomes and generate corresponding masks."""    
    # Generate filled regions corresponding to red- and blue-labeled liposomes.
    # Note that blue is a false color used to represent ATTO 647 fluorescence.
    filled_red = fill_holes_including_edge(image_oir[:, :, 0])
    filled_blue = fill_holes_including_edge(image_oir[:, :, 1])
    
    # Assign 1 to red-LUV regions, -1 to blue-LUV regions, and 0 elsewhere.
    red_or_blue = np.zeros_like(image_oir[:, :, 0], dtype = np.int8)

    # Assign blue first so that red-liposome signals detected as noise in the
    # blue channel are overwritten by the red-channel information.
    red_or_blue[filled_blue] = -1
    red_or_blue[filled_red] = 1

    labels= np.unique(masks)
    labels = labels[labels != 0]
    red_or_blue_values = ndi.mean(red_or_blue, labels=masks, index=labels)

    # Classify droplets using empirical thresholds.
    labels_red = labels[red_or_blue_values > 0.75]
    labels_blue = labels[red_or_blue_values < -0.75]
    data_red = data[data['label'].isin(labels_red)].copy()
    data_blue = data[data['label'].isin(labels_blue)].copy()

    # Relabel objects consecutively from 1 within each group.
    masks_red = np.zeros_like(masks)
    masks_blue = np.zeros_like(masks)
    masks_red_inner = None
    masks_blue_inner = None
    if masks_inner is not None:
        masks_red_inner = np.zeros_like(masks_inner)
        masks_blue_inner = np.zeros_like(masks_inner)

    for i, label in enumerate(labels_red):
        data_red.loc[data_red['label'] == label, 'label'] = i + 1
        masks_red[masks == label] = i+1
        if masks_inner is not None:
            masks_red_inner[masks_inner == label] = i+1
    for i, label in enumerate(labels_blue):
        data_blue.loc[data_blue['label'] == label, 'label'] = i + 1
        masks_blue[masks == label] = i+1
        if masks_inner is not None:
            masks_blue_inner[masks_inner == label] = i+1
    return data_red, data_blue, masks_red, masks_blue, masks_red_inner, masks_blue_inner

def shrink_by_distance(masks, target_area_ratio):
    """Generate inner masks by retaining pixels farthest from each object boundary."""
    inner_masks = np.zeros_like(masks)
    labels = np.unique(masks)
    labels = labels[labels != 0]
    for label in labels:
        mask = (masks == label)
        # Calculate the Euclidean distance from each pixel to the object boundary.
        dist = ndi.distance_transform_edt(mask)
        dist_values = dist[mask]
        if len(dist_values) == 0:
            continue
        # Determine the threshold required to retain the specified fraction of pixels closest to the object center.
        # Retain only pixels farther from the boundary than the threshold.
        percentile_val = (1.0 - target_area_ratio) * 100
        threshold = np.percentile(dist_values, percentile_val)
        inner_masks[(mask) & (dist > threshold)] = label
    return inner_masks

def measurement(image, masks, masks_processed = None, pixel_size_um = 0.248592227760895):
    """
    Measure object areas, centroids, and mean fluorescence intensities. 
    If masks_processed is provided, fluorescence is measured within the processed masks, 
    while object areas and centroids are calculated from the original masks.
    """
    labels = np.unique(masks)
    labels = labels[labels != 0]
    areas = [np.sum(masks == label) for label in labels]
    centroids = ndi.center_of_mass(image, labels=masks, index=labels)
    if masks_processed is not None:
        masks = masks_processed
    fluorescence_intensities = ndi.mean(image, labels=masks, index=labels)
    data = pd.DataFrame({
        'label': labels,
        'Area [um^2]' : [area * pixel_size_um**2 for area in areas],
        'Centroid_X' : [centroid[1] * pixel_size_um for centroid in centroids],
        'Centroid_Y' : [centroid[0] * pixel_size_um for centroid in centroids],
        'Fl_Int' : fluorescence_intensities
    })
    return data

def measurement_outside(image, masks, iterations=3):
    """
    Measure the mean background fluorescence outside dilated object masks.
    As a fallback, reduce the number of dilation iterations if no background pixels remain after dilation.
    """
    masks_dilated = ndi.binary_dilation(masks, structure=np.ones((20, 20)), iterations=iterations)
    fluorescence_intensity_outside = np.mean(image[~masks_dilated])
    if np.isnan(fluorescence_intensity_outside):
        while np.isnan(fluorescence_intensity_outside) and iterations > 1:
            iterations -= 1
            masks_dilated = ndi.binary_dilation(masks, structure=np.ones((20, 20)), iterations=iterations)
            fluorescence_intensity_outside = np.mean(image[~masks_dilated])
    return fluorescence_intensity_outside

def show_outline_label(image, masks, data, cmap='gray', ax=None, pixel_size_um=0.248592227760895):
    """Display object outlines and labels over the source image."""
    if ax is None:
        fig, ax = plt.subplots()
    outlines = utils.outlines_list(masks)
    ax.imshow(image, cmap=cmap)
    for o in outlines:
        ax.plot(o[:, 0], o[:, 1], color='y', lw=2, ls='--')
    for i, row in data.iterrows():
        ax.text(row['Centroid_X']/pixel_size_um, row['Centroid_Y']/pixel_size_um, int(row['label']), color='w', fontsize=8, fontweight='bold', ha='center', va='center')
    ax.axis('off')
    return ax


### Analysis

#### FITC-DEX and FAM-RNA exchange

In [8]:
# Channel assignments
# ATTO 565: red; ATTO 647: blue (false color); Cascade Blue: light blue; FITC/FAM: green; DIC: transmitted-light channel.
red_channel = 0
green_channel = 3
blue_channel = 1
cascade_blue_channel = 2

# Retain approximately the innermost 25% of each segmented droplet for fluorescence quantification.
target_area_ratio = 0.25

base_dir = os.path.join(main_dir, 'Microscope images for analysis')
dates_list = ['241024', '241029', '260409', '260413', '260415', '260609', '241129', '250207']
for date in dates_list:
    print(f"{date}, {dates_list.index(date)+1}/{len(dates_list)}(all)")
    image_for_analyze_dir = os.path.join(base_dir, date)
    # Create output directories.
    analyzed_dir = os.path.join(image_for_analyze_dir, f'Analyzed')
    os.makedirs(analyzed_dir, exist_ok=True)
    measurement_dir = os.path.join(analyzed_dir, 'Measurement')
    os.makedirs(measurement_dir, exist_ok=True)
    npy_dir = os.path.join(analyzed_dir, 'Numpy')
    os.makedirs(npy_dir, exist_ok=True)
    masks_dir = os.path.join(npy_dir, 'Masks')
    os.makedirs(masks_dir, exist_ok=True)
    segmentation_dir = os.path.join(analyzed_dir, 'Segmentation')
    os.makedirs(segmentation_dir, exist_ok=True)
    data_all = []
    outside = []
    
    # Select and sort OIR image files.
    image_list = os.listdir(image_for_analyze_dir)
    image_list = [image for image in image_list if image.endswith('.oir')]
    image_list.sort()

    for image in image_list:
        image_path = os.path.join(image_for_analyze_dir, image)
        image_oir = open_oir(image_path)
        
        # Generate an RGB image for Cellpose segmentation.
        # ATTO 565 and ATTO 647 signals are temporarily combined into the red channel, while Cascade Blue is assigned to the green channel.        
        red_ch = normalize_image(image_oir[:, :, red_channel])
        blue_ch = normalize_image(image_oir[:, :, blue_channel])
        seg_rgr = np.stack([red_ch + blue_ch, normalize_image(image_oir[:, :, cascade_blue_channel]), np.zeros_like(red_ch)], axis=2)
        seg_rgr = normalize_image(seg_rgr)
        
        # Segment droplets using Cellpose.
        # Retain only droplets with areas ≥100 pixels (approximately 6.2 µm²).
        # Generate inner masks for fluorescence quantification.
        masks, flows, styles = model.eval(seg_rgr, channels=[1, 2])
        masks_segmented = masks.copy()    
        masks = filter_masks_by_size(masks, size=100)
        masks_inner = shrink_by_distance(masks, target_area_ratio=target_area_ratio)
        
        # Quantify fluorescence intensities within the inner mask regions using the original images.
        filename = f'{image}_{date}'
        data = measurement(image_oir[:, :, green_channel], masks, masks_processed=masks_inner)
        data['CB_Fl_Int'] = measurement(image_oir[:, :, cascade_blue_channel], masks, masks_processed=masks_inner)['Fl_Int']
        data['Red_LUV'] = measurement(image_oir[:, :, red_channel], masks, masks_processed=masks_inner)['Fl_Int']
        data['Blue_LUV'] = measurement(image_oir[:, :, blue_channel], masks, masks_processed=masks_inner)['Fl_Int']
        data['Droplet'], data['Filename'] = 'Unknown', filename

        # Classify droplets into the Red and Blue groups.
        data_red, data_blue, masks_red, masks_blue, masks_red_inner, masks_blue_inner = divide_red_or_blue(image_oir, data, masks, masks_inner=masks_inner)
        data_red.loc[:, 'Droplet'] = 'Red'
        data_blue.loc[:, 'Droplet'] = 'Blue'
        data_all.extend([data_red, data_blue])

        # Measure the FITC/FAM background fluorescence outside the dilated masks.
        outside_mask = image_oir[:, :, cascade_blue_channel] > filters.threshold_otsu(image_oir[:, :, cascade_blue_channel])
        outside.append({
            'Outside': measurement_outside(image_oir[:, :, green_channel], outside_mask, iterations=3),
            'Filename': filename
        })

        # Save quantified data.
        data_red.to_csv(os.path.join(measurement_dir, f'{filename}_red.csv'), index=False)
        data_blue.to_csv(os.path.join(measurement_dir, f'{filename}_blue.csv'), index=False)
        # Save segmentation masks.
        np.save(os.path.join(masks_dir, f'{filename}_masks.npy'), masks_segmented)
        np.save(os.path.join(masks_dir, f'{filename}_masks_red.npy'), masks_red)
        np.save(os.path.join(masks_dir, f'{filename}_masks_blue.npy'), masks_blue)
        # Save segmentation results before and after inner-mask generation.
        red_green_blue_image = merge_rgb(image_oir, red_channel, green_channel, blue_channel)
        fig, ax= plt.subplots(2, 2, figsize=(10, 10))
        show_outline_label(red_green_blue_image, masks_red, data_red, ax=ax[0, 0])
        show_outline_label(red_green_blue_image, masks_blue, data_blue, ax=ax[0, 1])
        show_outline_label(red_green_blue_image, masks_red_inner, data_red, ax=ax[1, 0])
        show_outline_label(red_green_blue_image, masks_blue_inner, data_blue, ax=ax[1, 1])
        plt.tight_layout()
        # plt.show()
        fig.savefig(os.path.join(segmentation_dir, f'{filename}.png'))
        plt.close()

    # Combine and save measurements from all images.
    data_all = pd.concat(data_all, ignore_index=True)
    data_all.to_csv(os.path.join(analyzed_dir, 'data_all.csv'), index=False)
    outside = pd.DataFrame(outside)
    outside.to_csv(os.path.join(analyzed_dir, 'outside.csv'), index=False)

241024, 1/8(all)


Operating in headless mode - the original ImageJ will have limited functionality.


241029, 2/8(all)
260409, 3/8(all)
260413, 4/8(all)
260415, 5/8(all)
260609, 6/8(all)
241129, 7/8(all)
250207, 8/8(all)


#### TcRR and TcCRR via inter-droplet molecular diffusion

In [10]:
# This analysis follows essentially the same workflow as described above.

# Channel assignments
# FAM: green; ATTO 565: red; Cascade Blue: light blue; Cy5: blue (false color); DIC: transmitted-light channel.
red_channel = 1
green_channel = 0
blue_channel = 3
cascade_blue_channel = 2

# Retain approximately the innermost 25% of each segmented droplet for fluorescence quantification.
target_area_ratio = 0.25

base_dir = os.path.join(main_dir, 'Microscope images for analysis')
dates_list = ['241119', '241203', '241206', '250506', '251028', '251029', '251030', '251015', '251017', '251024', '260221', '260319']
for date in dates_list:
    print(f"{date}, {dates_list.index(date)+1}/{len(dates_list)}(all)")
    # Create output directories.
    image_for_analyze_dir = os.path.join(base_dir, date)
    analyzed_dir = os.path.join(image_for_analyze_dir, f'Analyzed')
    os.makedirs(analyzed_dir, exist_ok=True)
    measurement_dir = os.path.join(analyzed_dir, 'Measurement')
    os.makedirs(measurement_dir, exist_ok=True)
    npy_dir = os.path.join(analyzed_dir, 'Numpy')
    os.makedirs(npy_dir, exist_ok=True)
    masks_dir = os.path.join(npy_dir, 'Masks')
    os.makedirs(masks_dir, exist_ok=True)
    segmentation_dir = os.path.join(analyzed_dir, 'Segmentation')
    os.makedirs(segmentation_dir, exist_ok=True)
    data_all = []
    outside = []
    
    # Select and sort OIR image files.
    image_list = os.listdir(image_for_analyze_dir)
    image_list = [image for image in image_list if image.endswith('.oir')]
    image_list.sort()

    for image in image_list:
        image_path = os.path.join(image_for_analyze_dir, image)
        image_oir = open_oir(image_path)
        
        # Generate a composite image containing the ATTO 565 and Cascade Blue signals for Cellpose segmentation.
        for_segmentation = merge_rgb(image_oir, red_channel, None, cascade_blue_channel)
        masks, flows, styles = model.eval(for_segmentation, channels=[1, 3])
        masks_segmented = masks.copy()
        masks = filter_masks_by_size(masks, size=100)
        masks_inner = shrink_by_distance(masks, target_area_ratio=target_area_ratio)
        
        # Quantify fluorescence intensities within the inner mask regions using the original images.
        filename = f'{image}_{date}'
        data = measurement(image_oir[:, :, blue_channel], masks, masks_processed=masks_inner)
        data['FAM_Fl_Int'] = measurement(image_oir[:, :, green_channel], masks, masks_processed=masks_inner)['Fl_Int']
        data['CB_Fl_Int'] = measurement(image_oir[:, :, cascade_blue_channel], masks, masks_processed=masks_inner)['Fl_Int']
        data['Red_LUV'] = measurement(image_oir[:, :, red_channel], masks, masks_processed=masks_inner)['Fl_Int']
        data['Droplet'], data['Filename'] = 'Unknown', filename

        # Classify droplets into Green and NoGreen groups based on log-transformed FAM fluorescence intensities.
        data_green, data_nogreen, masks_green, masks_nogreen, masks_green_inner, masks_nogreen_inner = divide_green_or_not(data, masks, masks_inner=masks_inner)
        data_green.loc[:, 'Droplet'] = 'Green'
        data_nogreen.loc[:, 'Droplet'] = 'NoGreen'
        data_all.extend([data_green, data_nogreen])

        # Define the droplet-containing region using Cascade Blue fluorescence and measure Cy5 background outside its dilated mask.
        outside_mask = image_oir[:, :, cascade_blue_channel] > filters.threshold_otsu(image_oir[:, :, cascade_blue_channel])
        outside.append({
            'Outside': measurement_outside(image_oir[:, :, blue_channel], outside_mask, iterations=3),
            'Filename': filename
        })

        # Save quantified data.
        data_green.to_csv(os.path.join(measurement_dir, f'{filename}_green.csv'), index=False)
        data_nogreen.to_csv(os.path.join(measurement_dir, f'{filename}_nogreen.csv'), index=False)
        # Save segmentation masks.
        np.save(os.path.join(masks_dir, f'{filename}_masks.npy'), masks_segmented)
        np.save(os.path.join(masks_dir, f'{filename}_masks_green.npy'), masks_green)
        np.save(os.path.join(masks_dir, f'{filename}_masks_nogreen.npy'), masks_nogreen)
        
        # Save segmentation results before and after inner-mask generation.
        green_droplets = merge_rgb(image_oir, red_channel, green_channel, None)
        fig, ax= plt.subplots(2, 2, figsize=(10, 10))
        show_outline_label(green_droplets, masks_green, data_green, ax=ax[0, 0])
        show_outline_label(green_droplets, masks_nogreen, data_nogreen, ax=ax[0, 1])
        show_outline_label(green_droplets, masks_green_inner, data_green, ax=ax[1, 0])
        show_outline_label(green_droplets, masks_nogreen_inner, data_nogreen, ax=ax[1, 1])
        plt.tight_layout()
        # plt.show()
        fig.savefig(os.path.join(segmentation_dir, f'{filename}.png'))
        plt.close()

    # Combine and save measurements from all images.
    data_all = pd.concat(data_all, ignore_index=True)
    data_all.to_csv(os.path.join(analyzed_dir, 'data_all.csv'), index=False)
    outside = pd.DataFrame(outside)
    outside.to_csv(os.path.join(analyzed_dir, 'outside.csv'), index=False)

241119, 1/12(all)


C:\Users\kfrsj\anaconda3\envs\cellpose\lib\site-packages\sklearn\cluster\_kmeans.py:1382: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\kfrsj\anaconda3\envs\cellpose\lib\site-packages\sklearn\cluster\_kmeans.py:1382: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


241203, 2/12(all)


C:\Users\kfrsj\anaconda3\envs\cellpose\lib\site-packages\sklearn\cluster\_kmeans.py:1382: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


241206, 3/12(all)


...

C:\Users\kfrsj\anaconda3\envs\cellpose\lib\site-packages\sklearn\cluster\_kmeans.py:1382: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


260221, 11/12(all)


C:\Users\kfrsj\anaconda3\envs\cellpose\lib\site-packages\sklearn\cluster\_kmeans.py:1382: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


260319, 12/12(all)


C:\Users\kfrsj\anaconda3\envs\cellpose\lib\site-packages\sklearn\cluster\_kmeans.py:1382: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
